In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import re

# --- Step 1: Load CSV ---
df = pd.read_csv('job5.csv')  # Use your actual CSV file

# --- Step 2: Convert relative Date strings to actual dates ---
def convert_relative_date(value):
    today = datetime.today()
    value = str(value).strip().lower()

    if value in ["just now", "few hours ago", "today"]:
        return today.date()
    elif "day ago" in value:
        match = re.search(r"(\d+)", value)
        if match:
            days = int(match.group(1))
            return (today - timedelta(days=days)).date()
    return pd.NaT

df['Date'] = df['Date'].apply(convert_relative_date)

# --- Step 3: Convert Salary to numeric INR ---
def extract_salary(sal):
    if isinstance(sal, str) and "lacs" in sal.lower():
        nums = re.findall(r"[\d.]+", sal)
        if len(nums) == 2:
            return round((float(nums[0]) + float(nums[1])) / 2 * 100000)
        elif len(nums) == 1:
            return round(float(nums[0]) * 100000)
    return None

df['Salary'] = df['Salary'].apply(extract_salary)

# --- Step 4: Replace missing Salary with mean ---
mean_salary = df['Salary'].dropna().mean()
df['Salary'] = df['Salary'].fillna(round(mean_salary))

# --- Step 5: Drop remaining null values (if any) ---
df.dropna(inplace=True)

# --- Step 6: Categorize jobs based on Skills ---
def categorize_skills(skills):
    skills = str(skills).lower()
    if any(word in skills for word in [
        "android", "kotlin", "flutter", "swift", "ios", "mobile",
        "html", "css", "javascript", "react", "angular", "node",
        "django", "flask", "php", "typescript", "ui/ux", "figma", 
        "adobe xd", "design", "user experience", "user interface",
        "game development", "unity", "unreal engine", "game design"]):
        return "Web and Mobile App Developer"
    elif any(word in skills for word in [
        "database", "sql", "mysql", "postgresql", "mongodb", "oracle",
        "big data", "hadoop", "spark", "kafka", "etl", "data pipeline",
        "ai", "machine learning", "deep learning", "data science",
        "pandas", "numpy", "scikit-learn", "tensorflow", "pytorch",
        "nlp", "computer vision"]):
        return "Data Scientist"
    elif any(word in skills for word in [
        "aws", "azure", "cloud", "gcp", "devops", "kubernetes", 
        "docker", "terraform"]):
        return "Cloud Engineer"
    elif any(word in skills for word in [
        "blockchain", "ethereum", "solidity", "smart contracts", "web3",
        "cybersecurity", "ethical hacking", "penetration testing", 
        "network security", "forensics", "testing", "selenium", 
        "junit", "pytest", "automation testing", "manual testing"]):
        return "Cybersecurity Engineer"
    elif any(word in skills for word in [
        "networking", "ccna", "ccnp", "tcp/ip", "routing", "switching"]):
        return "Network Engineer"
    elif any(word in skills for word in [
        "c", "c++", "java", "python", "software development", 
        "algorithms", "data structures"]):
        return "Software Developer"
    else:
        return "Other"

df["Field"] = df["Skills"].apply(categorize_skills)

# --- Step 7: Save and print category counts ---
df.to_csv("preprocessed.csv", index=False)

category_counts = df["Field"].value_counts()
print("✅ Cleaned and classified data saved to 'finaljob_classified.csv'")
print("\n📊 Total counts per field:")
print(category_counts)


✅ Cleaned and classified data saved to 'finaljob_classified.csv'

📊 Total counts per field:
Field
Web and Mobile App Developer    59
Software Developer              18
Other                            1
Data Scientist                   1
Name: count, dtype: int64
